In [1]:
import lfox
import lfox.lattice as lat
import lfox.evolution.hmc as lhmc
import jax
import jax.numpy as jnp
import numpy as np

# Imports below require "dev" environment
import matplotlib.pyplot as plt
import lsqfit
import gvar as gv
import tqdm

# Double precision!
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_threefry_partitionable", True)


In [2]:
class ScalarAction(lhmc.Action):        

    @staticmethod
    @jax.jit
    def _S(fields, params):
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d()):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S
    
    # Exact force function instead of autodiff, for testing purposes
    @staticmethod
    @jax.jit
    def exact_force(fields, params):
        phi = fields[0]

        J = phi.nn_field(axis=0, shift=1) + phi.nn_field(axis=0, shift=-1)
        for ax in range(1,phi.d()):
            J += phi.nn_field(axis=ax, shift=1)
            J += phi.nn_field(axis=ax, shift=-1)

        F = -2 * params['kappa'] * J
        F += 2 * phi.F
        F += 4 * params['lambda'] * (phi.F**2 - 1) * phi.F

        return F

In [3]:
def show_live(verbose=False):
    if verbose:
        print("Live details: ")
        print(jax.live_arrays())
        print("------------------------------")
    
    print("Live arrays = ", len(jax.live_arrays()))

In [4]:
lat.SquareLattice(st_dims=((4,4)))
lat.HoneycombLattice(st_dims=((4,4)))
show_live()

Live arrays =  0


I0000 00:00:1705639493.447248       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [5]:
d = 3
Lat4 = lat.SquareLattice(st_dims=((4,)*d))
phi4 = lat.LatticeField(lattice=Lat4, F=1)
#phi4.F = np.ones_like(phi4.F)
#phi4.set_field(np.ones_like(phi4.F))

show_live()  # Expect 4: {phi4.F} and the three _bc_coords fields in Lat4.
print(jax.live_arrays())
print('===========')

S4 = ScalarAction(field_names=['phi'], params={'kappa': 0.18169, 'lambda': 1.3282})

print(phi4)
phi4.nn_field(1)
show_live()  # Expect 4 still...get 5?
print(jax.live_arrays())


Live arrays =  4
[Array([[[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]],

       [[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]], dtype=float64), Array([[[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]],

       [[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]],

       [[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]],

       [[0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3],
        [0, 1, 2, 3]]], dtype=int64), Array([[[0, 0, 0, 0],
        [1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]],

       [[0, 0, 0, 0],
        [1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 

In [6]:
print(len(phi4.bc))
print(phi4.d())
print(phi4)
print(Lat4 == Lat4)
print(hash(Lat4))
print(phi4 * phi4.nn_field(0))

3
3
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
True
5958266269407395088
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)


In [27]:

HMC = lhmc.HMCRewrite(
    action=S4,
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

HMC.evolve(
    fields={'phi': phi4},
    rng_key=jax.random.PRNGKey(42),
    warmup=True,
)

({'phi': LatticeField(
    lattice=SquareLattice(
      st_dims=(4, 4, 4),
      _dims=(4, 4, 4),
      _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
    ),
    F=f64[4,4,4],
    bc=(1, 1, 1),
    indices=()
  )},
 {'P_acc': Array(1.45656559, dtype=float64),
  'accept': Array(True, dtype=bool),
  'delta_H': Array(-0.37608133, dtype=float64)},
 Array([1832780943,  270669613], dtype=uint32))

In [30]:
%%timeit

rng_key = jax.random.PRNGKey(1841)
fields={'phi': phi4}
monitor = []

for _ in range(10000):
    fields, mon_step, rng_key = HMC.evolve(
        fields=fields,
        rng_key=rng_key,
        warmup=True
    )

    monitor.append(mon_step)

208 ms ± 484 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
show_live()

Live arrays =  3039


In [25]:
%%timeit

HMC = lhmc.HMCEvolver(
    action=S4,
    seed=72345,
    init_fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

show_live()  # Expect 3: {phi4, HMC.rng_key, HMC.pi_fields}

for _ in range(1000):
    HMC.evolve(warmup=False)

Live arrays =  3100


IndexError: index 1 is out of bounds for axis 0 with size 1

In [22]:
@jax.jit
def S2(fields, params):
        print("fields", fields)
        print("Params", params)
        phi = fields[0]
        S = phi**2
        
        for ax in range(phi.d()):
            S -= 2 * params['kappa'] * phi * phi.nn_field(ax)

        S += params['lambda'] * (phi**2 - 1)**2
        return S

S2([phi4], S4.params)

fields [LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)]
Params {'kappa': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>, 'lambda': Traced<ShapedArray(float64[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>}


LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)

In [23]:
print(phi4)
print(phi4**2)
print(S4._S([phi4], S4.params))
S4.S({'phi': phi4})

LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)
LatticeField(
  lattice=SquareLattice(
    st_dims=(4, 4, 4),
    _dims=(4, 4, 4),
    _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
  ),
  F=f64[4,4,4],
  bc=(1, 1, 1),
  indices=()
)


Array(-5.76896, dtype=float64)

In [43]:
jax.clear_caches()  #Fixes the leak?

HMC = lhmc.HMCEvolver(
    action=S4,
    seed=72345,
    init_fields={'phi': phi4},
    integrator=lhmc.LeapfrogIntegrator(eps=0.1, Nstep=10),
)

show_live()  # Expect 3: {phi4, HMC.rng_key, HMC.pi_fields}

HMC.evolve(warmup=False)

show_live()  # Expect 4: above plus new entry in field_chain

HMC.evolve(warmup=True)

show_live()  # Expect 5: above plus new entry in field_chain

#HMC.evolve(warmup=True)

#show_live()


Live arrays =  43
Live arrays =  45
Live arrays =  38


In [36]:
HMC.__dict__

{'integrator': LeapfrogIntegrator(eps=0.1, Nstep=10),
 'monitor': {'delta_H': [-0.25304516818601885, -1.1507334466190087],
  'P_acc': [1.287941449499816, 3.160510125274647],
  'accept': [True, True]},
 'traj_init': 0,
 'traj_chain': [0, 1, 2],
 'action': ScalarAction(
   field_names=['phi'],
   params={'kappa': 0.18169, 'lambda': 1.3282},
   sub_actions=[]
 ),
 'seed': 72345,
 'rng_key': Array([ 847442375, 2892676708], dtype=uint32),
 'fields': {'phi': LatticeField(
    lattice=SquareLattice(
      st_dims=(4, 4, 4),
      _dims=(4, 4, 4),
      _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
    ),
    F=f64[1,1,4,4,4],
    bc=(1, 1, 1),
    indices=()
  )},
 'save_freq': 1,
 'observables': None,
 'field_chain': {'phi': [LatticeField(
     lattice=SquareLattice(
       st_dims=(4, 4, 4),
       _dims=(4, 4, 4),
       _bc_coords=(i64[4,4,4], i64[4,4,4], i64[4,4,4])
     ),
     F=f64[4,4,4],
     bc=(1, 1, 1),
     indices=()
   ),
   LatticeField(
     lattice=SquareLattice(
       s